# Advanced Clustering: DBSCAN, Hierarchical & Gaussian Mixture Models

Beyond K-Means: density-based, hierarchical, and probabilistic approaches to clustering.

## 1. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.colors import ListedColormap
import seaborn as sns

from sklearn.datasets import make_blobs, make_moons, make_circles, make_classification
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering, SpectralClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import (
    silhouette_score, davies_bouldin_score, calinski_harabasz_score,
    adjusted_rand_score, normalized_mutual_info_score
)
from sklearn.neighbors import NearestNeighbors
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import pdist

sns.set_theme(style='whitegrid')
np.random.seed(42)
%matplotlib inline

## 2. Limitations of K-Means

K-Means is fast and simple but has several fundamental weaknesses:
- **Spherical clusters only**: assumes clusters are isotropic (circular in 2D)
- **Equal size bias**: tends to produce clusters of similar cardinality
- **Sensitive to outliers**: every point is assigned to a cluster, outliers pull centroids
- **Requires K upfront**: must know the number of clusters in advance
- **Initialisation dependent**: different seeds can yield different results

Let's illustrate these limitations.

In [ ]:
def plot_kmeans_failures():
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

    # 1) Non-spherical: moons
    X_moons, _ = make_moons(n_samples=300, noise=0.08, random_state=42)
    kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_moons)
    axes[0].scatter(X_moons[:, 0], X_moons[:, 1], c=labels, cmap='viridis', edgecolors='k', s=40)
    axes[0].set_title('K-Means on Moons (fails)', fontsize=12)

    # 2) Concentric circles
    X_circ, _ = make_circles(n_samples=300, noise=0.05, factor=0.5, random_state=42)
    labels = KMeans(n_clusters=2, random_state=42, n_init=10).fit_predict(X_circ)
    axes[1].scatter(X_circ[:, 0], X_circ[:, 1], c=labels, cmap='viridis', edgecolors='k', s=40)
    axes[1].set_title('K-Means on Circles (fails)', fontsize=12)

    # 3) Outliers
    rng = np.random.RandomState(42)
    X_blob, _ = make_blobs(n_samples=200, centers=1, cluster_std=1.2, random_state=42)
    outliers = rng.uniform(low=-8, high=8, size=(12, 2))
    X_out = np.vstack([X_blob, outliers])
    labels = KMeans(n_clusters=3, random_state=42, n_init=10).fit_predict(X_out)
    axes[2].scatter(X_out[:, 0], X_out[:, 1], c=labels, cmap='viridis', edgecolors='k', s=40)
    axes[2].set_title('K-Means + Outliers (forced split)', fontsize=12)

    plt.tight_layout()
    plt.show()

plot_kmeans_failures()

---
## 3. DBSCAN — Density-Based Spatial Clustering of Applications with Noise

DBSCAN groups points that are **densely packed** and marks points in low-density regions as **noise**.

### Key concepts
- **Core point**: has at least `min_samples` points within distance `eps` (including itself)
- **Border point**: within `eps` of a core point, but has fewer than `min_samples` neighbours
- **Noise point**: neither core nor border

### Parameters
- `eps` (ε): maximum distance between two points to be considered neighbours
- `min_samples`: minimum points to form a dense region

### Advantages
- Handles arbitrary-shaped clusters
- Automatically detects outliers (noise)
- No need to specify number of clusters

In [ ]:
# Helper to visualise core / border / noise
def dbscan_point_types():
    X, _ = make_moons(n_samples=200, noise=0.06, random_state=42)
    db = DBSCAN(eps=0.2, min_samples=5)
    labels = db.fit_predict(X)

    core_mask = np.zeros_like(labels, dtype=bool)
    core_mask[db.core_sample_indices_] = True
    noise_mask = labels == -1
    border_mask = ~core_mask & ~noise_mask

    fig, ax = plt.subplots(1, 1, figsize=(8, 5))
    ax.scatter(X[core_mask, 0], X[core_mask, 1], c='#2c7bb6', edgecolors='k',
               s=60, label='Core', alpha=0.85)
    ax.scatter(X[border_mask, 0], X[border_mask, 1], c='#fdae61', edgecolors='k',
               s=60, label='Border', alpha=0.85)
    ax.scatter(X[noise_mask, 0], X[noise_mask, 1], c='#d7191c', edgecolors='k',
               s=60, marker='x', label='Noise', alpha=0.85)
    ax.set_title('DBSCAN Point Types (core / border / noise)', fontsize=13)
    ax.legend(fontsize=11)
    plt.show()

dbscan_point_types()

### 3.1 Choosing ε — the k-distance graph

Plot the distance to the k-th nearest neighbour (k = `min_samples`) sorted in ascending order. The "elbow" gives a good ε value.

In [ ]:
def k_distance_graph(X, k=5):
    neigh = NearestNeighbors(n_neighbors=k)
    neigh.fit(X)
    distances, _ = neigh.kneighbors(X)
    k_dist = np.sort(distances[:, -1])

    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.plot(k_dist, lw=2)
    ax.axhline(y=k_dist[150], color='crimson', ls='--', label=f'elbow ≈ {k_dist[150]:.3f}')
    ax.set_xlabel('Points sorted by distance', fontsize=11)
    ax.set_ylabel(f'{k}-th Nearest Neighbour Distance', fontsize=11)
    ax.set_title(f'k-Distance Graph (k = {k}) — choose ε at the elbow', fontsize=12)
    ax.legend()
    plt.tight_layout()
    plt.show()

X_moons, _ = make_moons(n_samples=300, noise=0.07, random_state=42)
k_distance_graph(X_moons, k=5)

In [ ]:
# DBSCAN on moons & circles with correct eps
def dbscan_demo():
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

    datasets = [
        (make_moons(n_samples=300, noise=0.07, random_state=42), 'Moons', 0.22),
        (make_circles(n_samples=300, noise=0.05, factor=0.5, random_state=42), 'Concentric Circles', 0.18),
    ]

    for ax, ((X, _), title, eps) in zip(axes, datasets):
        labels = DBSCAN(eps=eps, min_samples=5).fit_predict(X)
        scatter = ax.scatter(X[:, 0], X[:, 1], c=labels, cmap='viridis', edgecolors='k', s=50)
        n_clusters = len(set(labels) - {-1})
        n_noise = np.sum(labels == -1)
        ax.set_title(f'{title}  |  clusters: {n_clusters}  noise: {n_noise}', fontsize=12)

    plt.tight_layout()
    plt.show()

dbscan_demo()

---
## 4. Hierarchical Clustering

Builds a tree (dendrogram) of nested clusters.

- **Agglomerative** (bottom-up): start with each point as its own cluster, merge closest pairs.
- **Divisive** (top-down): start with one big cluster, recursively split.

### Linkage criteria
| Linkage | Distance between clusters | Behaviour |
|---------|--------------------------|-----------|
| Single  | Minimum pairwise distance | Can chain, produces long thin clusters |
| Complete| Maximum pairwise distance | Compact clusters, sensitive to outliers |
| Average | Average pairwise distance | Balance between single & complete |
| Ward    | Increase in sum of squares | Produces spherical clusters, similar to K-Means |

In [ ]:
# Compare linkage criteria on blobs
X_blobs, y_blobs = make_blobs(n_samples=200, centers=3, cluster_std=1.5, random_state=42)

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
linkages = ['single', 'complete', 'average', 'ward']

for ax, link in zip(axes, linkages):
    model = AgglomerativeClustering(n_clusters=3, linkage=link)
    labels = model.fit_predict(X_blobs)
    ax.scatter(X_blobs[:, 0], X_blobs[:, 1], c=labels, cmap='Set2', edgecolors='k', s=45)
    ax.set_title(f'Linkage: {link}', fontsize=12)

plt.tight_layout()
plt.show()

### 4.1 Dendrograms

A dendrogram shows the merge/split history. The **height** of each join indicates the distance at which clusters were merged. Cutting at a chosen height gives a flat clustering.

**How to read**:
- Leaves = individual data points
- Branches = merged clusters
- Vertical axis = dissimilarity (distance)
- Horizontal cut = choose number of clusters

In [ ]:
# Dendrogram with cut line
X_blobs_small, _ = make_blobs(n_samples=60, centers=3, cluster_std=1.2, random_state=42)

Z = linkage(X_blobs_small, method='ward')

fig, ax = plt.subplots(figsize=(12, 5.5))
dendrogram(Z, ax=ax, leaf_rotation=90, leaf_font_size=9)
ax.axhline(y=18, color='crimson', ls='--', lw=2, label='Cut at height = 18 → 3 clusters')
ax.set_title('Dendrogram — Ward Linkage', fontsize=13)
ax.set_ylabel('Distance')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print(f'Clusters from flat cut: {np.unique(fcluster(Z, t=18, criterion="distance"))}')

---
## 5. Gaussian Mixture Models (GMM)

GMM is a **soft clustering** algorithm — each point belongs to every cluster with a certain **probability** (responsibility).

### Expectation-Maximisation (EM) algorithm
1. **E-step**: compute responsibilities (posterior probabilities) given current parameters
2. **M-step**: update means, covariances and mixing weights to maximise likelihood

### Covariance type controls cluster shape
- `full`: each cluster has its own general covariance (ellipsoidal, any orientation)
- `tied`: all clusters share the same covariance
- `diag`: diagonal covariance (axes-aligned ellipses)
- `spherical`: circular clusters

### Choosing K: BIC / AIC
**Bayesian Information Criterion (BIC)** and **Akaike Information Criterion (AIC)** balance model fit with complexity. Lower is better.

In [ ]:
# GMM with BIC / AIC for model selection
X_varied, _ = make_blobs(n_samples=500, centers=3, cluster_std=[1.0, 2.5, 0.8], random_state=42)

bics, aics = [], []
Ks = range(1, 9)
for k in Ks:
    gmm = GaussianMixture(n_components=k, covariance_type='full', random_state=42)
    gmm.fit(X_varied)
    bics.append(gmm.bic(X_varied))
    aics.append(gmm.aic(X_varied))

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(Ks, bics, 'o-', label='BIC', lw=2)
ax.plot(Ks, aics, 's--', label='AIC', lw=2)
ax.axvline(x=3, color='grey', ls=':', alpha=0.6)
ax.set_xlabel('Number of components', fontsize=11)
ax.set_ylabel('Score', fontsize=11)
ax.set_title('BIC / AIC for GMM — choose K at elbow', fontsize=12)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# GMM probability contours
def plot_gmm_contours():
    X, _ = make_blobs(n_samples=400, centers=3, cluster_std=1.3, random_state=42)
    gmm = GaussianMixture(n_components=3, covariance_type='full', random_state=42)
    gmm.fit(X)
    labels = gmm.predict(X)
    probs = gmm.predict_proba(X).max(axis=1)

    xx, yy = np.meshgrid(np.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, 100),
                         np.linspace(X[:, 1].min() - 1, X[:, 1].max() + 1, 100))
    Z = -gmm.score_samples(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    fig, ax = plt.subplots(figsize=(8, 6))
    scatter = ax.scatter(X[:, 0], X[:, 1], c=labels, cmap='Set2',
                         edgecolors='k', s=50, alpha=0.8)
    ax.contour(xx, yy, Z, levels=8, cmap='viridis', alpha=0.5)
    ax.set_title('GMM Clustering with Density Contours', fontsize=13)
    plt.tight_layout()
    plt.show()

    print('\nCluster assignments (first 10):', labels[:10])
    print('Max probability (first 10):     ', np.round(probs[:10], 3))

plot_gmm_contours()

---
## 6. Comparing All Methods on Synthetic Datasets

In [ ]:
# Comparison grid: K-Means, DBSCAN, Agglomerative, GMM on 4 datasets
datasets = [
    ('Blobs', *make_blobs(n_samples=300, centers=3, cluster_std=1.0, random_state=42)),
    ('Moons', *make_moons(n_samples=300, noise=0.08, random_state=42)),
    ('Circles', *make_circles(n_samples=300, noise=0.05, factor=0.5, random_state=42)),
    ('Varied', *make_blobs(n_samples=300, centers=3, cluster_std=[1.0, 2.5, 0.6], random_state=42)),
]

methods = [
    ('K-Means (K=3)', lambda X: KMeans(n_clusters=3, random_state=42, n_init=10).fit_predict(X)),
    ('DBSCAN',        lambda X: DBSCAN(eps=0.25, min_samples=5).fit_predict(X)),
    ('Agglom. (ward)',lambda X: AgglomerativeClustering(n_clusters=3, linkage='ward').fit_predict(X)),
    ('GMM (K=3)',     lambda X: GaussianMixture(n_components=3, covariance_type='full', random_state=42).fit_predict(X)),
]

fig, axes = plt.subplots(len(datasets), len(methods), figsize=(16, 13))

for i, (dname, X, y) in enumerate(datasets):
    for j, (mname, mfunc) in enumerate(methods):
        ax = axes[i, j]
        labels = mfunc(X)
        ax.scatter(X[:, 0], X[:, 1], c=labels, cmap='Set2', edgecolors='k', s=30)
        if i == 0:
            ax.set_title(mname, fontsize=11)
        if j == 0:
            ax.set_ylabel(dname, fontsize=11, fontweight='bold')
        ax.set_xticks([])
        ax.set_yticks([])

plt.suptitle('Clustering Methods Comparison', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

---
## 7. Cluster Validation Metrics

| Metric | Range | Best | Needs truth | Notes |
|--------|-------|------|-------------|-------|
| **Silhouette Score** | [-1, 1] | High | No | Measures cohesion vs separation |
| **Davies-Bouldin Index** | [0, ∞) | Low | No | Avg similarity between each cluster & its most similar |
| **Calinski-Harabasz Index** | [0, ∞) | High | No | Ratio of between-cluster / within-cluster dispersion |
| **Adjusted Rand Index (ARI)** | [-1, 1] | 1 | Yes | Measures similarity to ground truth (chance-adjusted) |
| **Normalised Mutual Info (NMI)** | [0, 1] | 1 | Yes | Information-theoretic agreement |

In [ ]:
# Validation metrics bar chart on Blobs dataset
X_val, y_val = make_blobs(n_samples=400, centers=3, cluster_std=1.2, random_state=42)

clusterers = {
    'K-Means': KMeans(n_clusters=3, random_state=42, n_init=10).fit_predict(X_val),
    'DBSCAN': DBSCAN(eps=0.35, min_samples=5).fit_predict(X_val),
    'Agglom. (ward)': AgglomerativeClustering(n_clusters=3, linkage='ward').fit_predict(X_val),
    'GMM': GaussianMixture(n_components=3, covariance_type='full', random_state=42).fit_predict(X_val),
}

metrics_names = ['Silhouette\n(higher)', 'Davies-Bouldin\n(lower)', 'Calinski-Harabasz\n(higher)', 'ARI', 'NMI']

def compute_all_metrics(labels, y_true):
    if len(set(labels)) < 2:
        return [np.nan] * 5
    sil = silhouette_score(X_val, labels)
    db = davies_bouldin_score(X_val, labels)
    ch = calinski_harabasz_score(X_val, labels)
    ari = adjusted_rand_score(y_true, labels)
    nmi = normalized_mutual_info_score(y_true, labels)
    return [sil, db, ch, ari, nmi]

scores = {name: compute_all_metrics(lab, y_val) for name, lab in clusterers.items()}

fig, axes = plt.subplots(1, 5, figsize=(18, 4.5))

for idx, (ax, mname) in enumerate(zip(axes, metrics_names)):
    names = list(scores.keys())
    vals = [scores[n][idx] for n in names]
    colors = ['#2c7bb6', '#fdae61', '#abd9e9', '#d7191c']
    bars = ax.bar(names, vals, color=colors, edgecolor='grey')
    ax.set_title(mname, fontsize=11)
    ax.tick_params(axis='x', rotation=20)
    for bar, v in zip(bars, vals):
        if not np.isnan(v):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                    f'{v:.2f}', ha='center', va='bottom', fontsize=8)

plt.suptitle('Cluster Validation Metrics Comparison', fontsize=13)
plt.tight_layout()
plt.show()

---
## 8. Real-World Example: Customer Segmentation

We apply all four methods to a synthetic customer dataset and compare results.

In [ ]:
np.random.seed(42)
n_customers = 600

df = pd.DataFrame({
    'Annual_Income': np.concatenate([
        np.random.normal(35, 8, 200),
        np.random.normal(65, 10, 200),
        np.random.normal(100, 15, 200),
    ]),
    'Spending_Score': np.concatenate([
        np.random.normal(80, 10, 200),
        np.random.normal(50, 12, 200),
        np.random.normal(25, 8, 200),
    ]),
})
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)

results = {}
results['K-Means'] = KMeans(n_clusters=3, random_state=42, n_init=10).fit_predict(X_scaled)
results['DBSCAN'] = DBSCAN(eps=0.4, min_samples=8).fit_predict(X_scaled)
results['Agglom.'] = AgglomerativeClustering(n_clusters=3, linkage='ward').fit_predict(X_scaled)
results['GMM'] = GaussianMixture(n_components=3, covariance_type='full', random_state=42).fit_predict(X_scaled)

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
for ax, (name, labels) in zip(axes, results.items()):
    ax.scatter(df['Annual_Income'], df['Spending_Score'],
               c=labels, cmap='Set2', edgecolors='k', s=40)
    ax.set_xlabel('Annual Income (k$)')
    ax.set_ylabel('Spending Score')
    ax.set_title(name, fontsize=12)

plt.suptitle('Customer Segmentation — All Methods', fontsize=13)
plt.tight_layout()
plt.show()

---
## 9. Spectral Clustering (Brief Introduction)

**Spectral clustering** uses the eigenvalues (spectrum) of the similarity matrix to reduce dimensionality before clustering.

- Works well on non-convex / connected structures (e.g. circles, moons)
- Based on graph cut theory — treats data as a graph and finds partitions with minimal edge cuts
- Key parameter: `affinity` (rbf, nearest_neighbors) and `n_clusters`
- Internally performs eigen-decomposition of the Laplacian matrix, then runs K-Means on the low-dimensional embedding

In [ ]:
X_circ_spec, _ = make_circles(n_samples=300, noise=0.05, factor=0.5, random_state=42)

spec = SpectralClustering(n_clusters=2, affinity='nearest_neighbors', random_state=42)
labels_spec = spec.fit_predict(X_circ_spec)

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(X_circ_spec[:, 0], X_circ_spec[:, 1], c=labels_spec, cmap='Set2', edgecolors='k', s=50)
ax.set_title('Spectral Clustering on Concentric Circles', fontsize=12)
plt.tight_layout()
plt.show()

---
## 10. Summary

| Method | Shape | Outliers | K known? | Soft? | Speed |
|--------|-------|----------|----------|-------|-------|
| K-Means | Spherical only | No | Yes | No | Very fast |
| DBSCAN | Arbitrary | **Yes** | No | No | Fast |
| Agglomerative | Depends on linkage | No | Optional | No | Medium |
| GMM | Ellipsoidal | Partial | Yes | **Yes** | Medium |
| Spectral | Arbitrary | No | Yes | No | Medium (eigen-decomp) |

**Choose DBSCAN** when clusters have arbitrary shapes and you expect noise.
**Choose Agglomerative** when you want a hierarchical view (dendrogram).
**Choose GMM** when you need soft assignments or elliptical clusters of different sizes.
**Choose Spectral** for strongly connected / manifold structures.

---
## Practice Exercises

1. **DBSCAN parameter tuning**  
   Generate `make_circles(n_samples=500, noise=0.08, factor=0.4)` and find the best `eps` and `min_samples` that recover both rings exactly (no noise points). Start with a k-distance graph.

2. **Dendrogram interpretation**  
   Create a dataset with 4 well-separated blobs (each n=40). Plot the dendrogram with `method='single'` and `method='complete'`. Explain why single linkage creates a different tree structure.

3. **GMM covariance types**  
   Use the "Varied" blobs from section 6. Fit GMMs with `covariance_type='full'`, `'diag'`, `'spherical'`, and `'tied'`. Plot clustering results side by side and report BIC for each. Which covariance type performs best and why?

4. **Outlier detection with DBSCAN**  
   Generate `make_blobs(n_samples=300, centers=2, cluster_std=0.8)` and add 20 random uniform outliers in the range [-8, 8]. Use DBSCAN to cluster and identify which points are labelled as noise (-1). Compare with K-Means (K=2).

5. **Soft clustering visualisation**  
   Fit a GMM with 3 components on the customer segmentation data. Instead of hard assignments, plot the data coloured by the **uncertainty** (1 - max probability). Which customers are hardest to assign?